# <font color="#418FDE" size="6.5" uppercase>**Persistence Deployment**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Choose appropriate persistence formats for scikit-learn models and pipelines. 
- Save and load fitted artifacts while considering security and environment compatibility. 
- Validate persisted or exported models after deployment-style loading. 


## **1. Persistence Formats**

### **1.1. Choosing Persistence Formats**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_01_01.jpg?v=1784044458" width="250">



>* Match format to deployment needs
>* Save full pipelines to avoid mismatches

>* Pickle and joblib suit controlled Python deployments
>* Use export formats for cross-platform serving

>* Match format to deployment constraints
>* Plan versions, metadata, access, and validation



### **1.2. Pickle Persistence**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_01_02.jpg?v=1784044460" width="250">



>* Save fitted pipelines as Python objects
>* Reload workflows without rebuilding each step

>* Flexible for many Python-based pipeline components
>* Requires matching code and dependency versions

>* Use pickle only in trusted Python environments
>* Track dependencies and avoid untrusted files



In [ ]:
#@title Python Code - Pickle Persistence

# This example saves a fitted pipeline.
# Pickle persistence keeps Python object structure.
# Reloading confirms deployment-style predictions still match.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import joblib



### **1.3. Joblib Persistence**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_01_03.jpg?v=1784044462" width="250">



>* Efficiently saves array-heavy scikit-learn artifacts
>* Preserves full pipelines for consistent deployment

>* Efficient storage for large array-based models
>* Best for compatible Python deployment environments

>* Load joblib files only from trusted sources
>* Track environment metadata for compatibility



In [ ]:
#@title Python Code - Joblib Persistence

# Save a fitted pipeline with joblib.
# Reload it like a deployment service.
# Compare predictions before and after loading.

import io

import joblib
import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Use a small packaged dataset for a fast example.
iris = load_iris()
features = iris.data
target = iris.target

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42, stratify=target
)

# Fit preprocessing and model together in one artifact.
pipeline = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=200, random_state=42)
)
pipeline.fit(X_train, y_train)

# Persist the fitted pipeline in memory using joblib.
artifact_buffer = io.BytesIO()
joblib.dump(pipeline, artifact_buffer)
artifact = artifact_buffer.getvalue()
loaded_pipeline = joblib.load(io.BytesIO(artifact))

# Compare predictions from the original and loaded artifacts.
original_predictions = pipeline.predict(X_test)
loaded_predictions = loaded_pipeline.predict(X_test)
match = np.array_equal(original_predictions, loaded_predictions)

# Print concise deployment-style validation results.
accuracy = accuracy_score(y_test, loaded_predictions)
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Serialized artifact size: {len(artifact)} bytes")
print(f"Loaded pipeline accuracy: {accuracy:.3f}")
print(f"Predictions match after loading: {match}")



## **2. Safe Model Artifacts**

### **2.1. Cloudpickle Serialization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_02_01.jpg?v=1784044454" width="250">



>* Serialize complex pipelines with custom Python logic
>* Reload complete artifacts for deployment scoring

>* Treat cloudpickle files as executable artifacts
>* Load only trusted, traceable model files

>* Match deployment environment to training setup
>* Validate loaded models with known examples



In [ ]:
#@title Python Code - Cloudpickle Serialization

# Demonstrate cloudpickle-style persistence with a fitted pipeline.
# Use sklearn serialization without writing local files.
# Validate loaded predictions against original predictions.

import numpy as np
import sklearn
import cloudpickle
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Validate the basic dataset shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and target labels must match.")

# Split data so preprocessing is fitted only on training data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Define a small custom preprocessing function.
def add_petal_ratio(features):
    ratio = features[:, 2] / features[:, 3]
    return np.column_stack((features, ratio))

# Build one pipeline containing custom logic and a classifier.
pipeline = Pipeline(
    steps=[
        ("custom_ratio", FunctionTransformer(add_petal_ratio)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

# Fit the complete artifact before serializing it.
pipeline.fit(X_train, y_train)
original_predictions = pipeline.predict(X_test)

# Serialize to bytes using cloudpickle.
artifact_bytes = cloudpickle.dumps(pipeline)
loaded_pipeline = cloudpickle.loads(artifact_bytes)

# Validate that deployment-style loading preserves predictions.
loaded_predictions = loaded_pipeline.predict(X_test)
predictions_match = np.array_equal(original_predictions, loaded_predictions)

# Report concise checks that matter after loading.
accuracy = accuracy_score(y_test, loaded_predictions)
print("scikit-learn version:", sklearn.__version__)
print("Serialized artifact size in bytes:", len(artifact_bytes))
print("Loaded predictions match original:", predictions_match)
print("Loaded pipeline test accuracy:", round(accuracy, 3))



### **2.2. Trusted skops Loading**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_02_02.jpg?v=1784044452" width="250">



>* Inspect artifacts before loading models
>* Allow only expected, trusted components

>* Inspect artifact types before loading
>* Approve expected components, investigate unfamiliar ones

>* Check artifact dependencies against production environment
>* Validate trusted models in staging first



### **2.3. Artifact Trust**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_02_03.jpg?v=1784044456" width="250">



>* Treat artifacts as production assets
>* Verify provenance before loading models

>* Store artifacts securely with clear metadata
>* Verify lineage and environment compatibility first

>* Verify artifacts before and after loading
>* Trust requires provenance, integrity, and behavior



## **3. Deployment Validation**

### **3.1. ONNX Export**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_03_01.jpg?v=1784044448" width="250">



>* ONNX enables portable model deployment
>* Validate exported behavior against original model

>* Compare original and ONNX predictions realistically
>* Validate preprocessing, tolerances, and deployment safety

>* Verify schema, types, ordering, and outputs
>* Test production loading and record validation evidence



### **3.2. Environment Reproduction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_03_02.jpg?v=1784044446" width="250">



>* Model behavior depends on software versions
>* Recreate runtime conditions before validating predictions

>* Package models with complete runtime details
>* Prevent mismatches from causing invalid predictions

>* Compare outputs across reproduced deployment environments
>* Test realistic cases and investigate discrepancies



In [ ]:
#@title Python Code - Environment Reproduction

# Validate a loaded model under reproduced conditions.
# Compare predictions before and after persistence.
# Confirm versions and outputs match closely.

import io

import joblib
import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic dataset shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and target labels must match.")

# Split data to mimic training and deployment records.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Fit one pipeline that includes preprocessing and modeling.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Save the fitted artifact into bytes using joblib utilities.
artifact_buffer = io.BytesIO()
joblib.dump(model, artifact_buffer)
artifact_bytes = artifact_buffer.getvalue()
loaded_model = joblib.load(io.BytesIO(artifact_bytes))

# Compare original and deployment-style loaded predictions.
original_predictions = model.predict(X_test)
loaded_predictions = loaded_model.predict(X_test)
match_rate = np.mean(original_predictions == loaded_predictions)

# Compare probabilities with a small numerical tolerance.
original_probabilities = model.predict_proba(X_test)[:, 1]
loaded_probabilities = loaded_model.predict_proba(X_test)[:, 1]
max_probability_gap = np.max(np.abs(original_probabilities - loaded_probabilities))

# Print concise validation evidence for deployment review.
print("scikit-learn version:", sklearn.__version__)
print("Test accuracy:", round(accuracy_score(y_test, loaded_predictions), 3))
print("Prediction match rate:", round(float(match_rate), 3))
print("Maximum probability gap:", round(float(max_probability_gap), 12))
print("Environment reproduction check passed:", bool(match_rate == 1.0))



### **3.3. Post Deployment Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_A/image_03_03.jpg?v=1784044450" width="250">



>* Test loaded models on known validation inputs
>* Compare outputs to catch deployment changes

>* Validate the complete prediction pipeline
>* Test realistic inputs and edge cases

>* Check speed, memory, errors, and output format
>* Track metadata, logging, monitoring, and reliability



In [ ]:
#@title Python Code - Post Deployment Checks

# Validate a loaded model after deployment.
# Compare predictions against trusted reference outputs.
# Confirm preprocessing and probabilities still match.

import numpy as np
import sklearn
from sklearn.datasets import load_iris

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Check the basic data contract before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and target labels must match.")

# Split data so validation records are not used for fitting.
X_train, X_check, y_train, y_check = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Fit one pipeline containing preprocessing and the classifier.
model_before_deployment = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=200, random_state=42)
)
model_before_deployment.fit(X_train, y_train)

# Store trusted outputs from the predeployment model.
reference_labels = model_before_deployment.predict(X_check)
reference_probabilities = model_before_deployment.predict_proba(X_check)

# Simulate deployment loading by reusing the fitted artifact object.
loaded_deployment_artifact = model_before_deployment
loaded_labels = loaded_deployment_artifact.predict(X_check)
loaded_probabilities = loaded_deployment_artifact.predict_proba(X_check)

# Compare labels exactly and probabilities within numeric tolerance.
labels_match = np.array_equal(reference_labels, loaded_labels)
probabilities_match = np.allclose(
    reference_probabilities, loaded_probabilities, atol=1e-12
)

# Summarize the post deployment checks clearly.
accuracy = accuracy_score(y_check, loaded_labels)
max_probability_difference = np.max(
    np.abs(reference_probabilities - loaded_probabilities)
)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Validation records checked: {len(X_check)}")
print(f"Loaded artifact accuracy: {accuracy:.3f}")
print(f"Predicted labels match reference: {labels_match}")
print(f"Class probabilities match tolerance: {probabilities_match}")
print(f"Largest probability difference: {max_probability_difference:.2e}")



# <font color="#418FDE" size="6.5" uppercase>**Persistence Deployment**</font>


In this lecture, you learned to:
- Choose appropriate persistence formats for scikit-learn models and pipelines. 
- Save and load fitted artifacts while considering security and environment compatibility. 
- Validate persisted or exported models after deployment-style loading. 

<font color='yellow'>Congratulations on completing this course!</font>